In [1]:
# Cell 1: Import libraries and load stock data
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

print(" Libraries imported!")

# Load stock price data (FIX: ensure numeric columns)
stocks = ['AAPL', 'TSLA', 'JPM', 'AMZN', 'MSFT']
stock_data = {}

for ticker in stocks:
    df = pd.read_csv(f'../data/{ticker}_prices.csv', index_col=0, parse_dates=True)
    
    # Convert all price/volume columns to numeric
    numeric_cols = ['Open', 'High', 'Low', 'Close', 'Adj Close', 'Volume']
    for col in numeric_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')
    
    # Drop any rows with NaN after conversion
    df = df.dropna()
    
    stock_data[ticker] = df
    print(f"✅ Loaded {ticker}: {len(df)} days")

print("\n Starting Feature Engineering!")

 Libraries imported!
✅ Loaded AAPL: 500 days
✅ Loaded TSLA: 500 days
✅ Loaded JPM: 500 days
✅ Loaded AMZN: 500 days
✅ Loaded MSFT: 500 days

 Starting Feature Engineering!


In [2]:
print("\n CALCULATING TECHNICAL INDICATORS")
print("="*60)

def add_technical_indicators(df):
    """Add technical analysis indicators to stock dataframe"""
    
    # Make a copy
    df = df.copy()
    
    # 1. RETURNS
    df['returns'] = df['Close'].pct_change()
    df['log_returns'] = np.log(df['Close'] / df['Close'].shift(1))
    
    # 2. MOVING AVERAGES
    df['MA_7'] = df['Close'].rolling(window=7).mean()
    df['MA_21'] = df['Close'].rolling(window=21).mean()
    df['MA_50'] = df['Close'].rolling(window=50).mean()
    
    # 3. EXPONENTIAL MOVING AVERAGES
    df['EMA_12'] = df['Close'].ewm(span=12, adjust=False).mean()
    df['EMA_26'] = df['Close'].ewm(span=26, adjust=False).mean()
    
    # 4. MACD (Moving Average Convergence Divergence)
    df['MACD'] = df['EMA_12'] - df['EMA_26']
    df['MACD_signal'] = df['MACD'].ewm(span=9, adjust=False).mean()
    
    # 5. BOLLINGER BANDS
    df['BB_middle'] = df['Close'].rolling(window=20).mean()
    df['BB_std'] = df['Close'].rolling(window=20).std()
    df['BB_upper'] = df['BB_middle'] + (2 * df['BB_std'])
    df['BB_lower'] = df['BB_middle'] - (2 * df['BB_std'])
    df['BB_width'] = df['BB_upper'] - df['BB_lower']
    
    # 6. RSI (Relative Strength Index)
    delta = df['Close'].diff()
    gain = (delta.where(delta > 0, 0)).rolling(window=14).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(window=14).mean()
    rs = gain / loss
    df['RSI'] = 100 - (100 / (1 + rs))
    
    # 7. VOLATILITY
    df['volatility'] = df['returns'].rolling(window=21).std()
    
    # 8. VOLUME FEATURES
    df['volume_change'] = df['Volume'].pct_change()
    df['volume_MA_7'] = df['Volume'].rolling(window=7).mean()
    
    # 9. PRICE POSITION
    df['high_low_pct'] = (df['High'] - df['Low']) / df['Close']
    df['close_open_pct'] = (df['Close'] - df['Open']) / df['Open']
    
    return df

# Apply technical indicators to all stocks
stock_data_with_features = {}

for ticker in stocks:
    print(f"\ Processing {ticker}...")
    df = add_technical_indicators(stock_data[ticker])
    stock_data_with_features[ticker] = df
    print(f"   Added {df.shape[1] - stock_data[ticker].shape[1]} technical indicators")

print("\n Technical indicators calculated for all stocks!")

# Show sample
print("\n Sample features for AAPL:")
sample_cols = ['Close', 'returns', 'MA_7', 'MA_21', 'RSI', 'MACD', 'volatility']
print(stock_data_with_features['AAPL'][sample_cols].tail())


 CALCULATING TECHNICAL INDICATORS
\ Processing AAPL...
   Added 20 technical indicators
\ Processing TSLA...
   Added 20 technical indicators
\ Processing JPM...
   Added 20 technical indicators
\ Processing AMZN...
   Added 20 technical indicators
\ Processing MSFT...
   Added 20 technical indicators

 Technical indicators calculated for all stocks!

 Sample features for AAPL:
                 Close   returns        MA_7       MA_21        RSI      MACD  \
Date                                                                            
2026-11-02  338.320007 -0.018309  344.918575  341.904288  46.588040  3.020591   
2026-11-03  331.899994 -0.018976  342.657144  341.857144  38.469566  1.876527   
2026-11-04  349.510010  0.053058  343.457145  342.613335  55.811797  2.363584   
2026-11-05  344.170013 -0.015279  343.038574  342.970478  50.518339  2.292263   
2026-11-06  342.369995 -0.005230  342.830004  343.063334  49.591510  2.066671   

            volatility  
Date                    


In [3]:
print("\n🔗 MERGING SENTIMENT WITH PRICE DATA")
print("="*60)

stock_data_final = {}

for ticker in stocks:
    print(f"\n📊 Processing {ticker}...")
    
    # Get price data with technical indicators
    df = stock_data_with_features[ticker].copy()
    
    # Try to load sentiment data
    try:
        sentiment_df = pd.read_csv(f'../data/{ticker}_daily_sentiment.csv')
        sentiment_df['date'] = pd.to_datetime(sentiment_df['date'])
        sentiment_df.set_index('date', inplace=True)
        
        # Merge on date
        df = df.join(sentiment_df, how='left')
        
        # Forward fill sentiment (if no news, use previous day's sentiment)
        sentiment_cols = ['avg_sentiment', 'sentiment_std', 'news_count', 
                         'avg_positive', 'avg_negative', 'avg_neutral']
        df[sentiment_cols] = df[sentiment_cols].fillna(method='ffill')
        
        # Fill remaining NaNs with neutral (0)
        df['avg_sentiment'] = df['avg_sentiment'].fillna(0)
        df['sentiment_std'] = df['sentiment_std'].fillna(0)
        df['news_count'] = df['news_count'].fillna(0)
        df['avg_positive'] = df['avg_positive'].fillna(0.33)
        df['avg_negative'] = df['avg_negative'].fillna(0.33)
        df['avg_neutral'] = df['avg_neutral'].fillna(0.34)
        
        print(f"   ✅ Merged sentiment data")
        print(f"   Days with news: {(df['news_count'] > 0).sum()}")
        
    except FileNotFoundError:
        print(f"   ⚠️ No sentiment data found - using price features only")
        # Add placeholder sentiment columns (all zeros)
        df['avg_sentiment'] = 0
        df['sentiment_std'] = 0
        df['news_count'] = 0
        df['avg_positive'] = 0.33
        df['avg_negative'] = 0.33
        df['avg_neutral'] = 0.34
    
    stock_data_final[ticker] = df

print("\n✅ Data merging complete!")

# Show final feature count
sample_ticker = stocks[0]
print(f"\n📊 Final feature count for {sample_ticker}: {stock_data_final[sample_ticker].shape[1]} features")
print(f"   Features: {list(stock_data_final[sample_ticker].columns)}")


🔗 MERGING SENTIMENT WITH PRICE DATA

📊 Processing AAPL...


TypeError: NDFrame.fillna() got an unexpected keyword argument 'method'

In [ ]:
import sys, traceback
ip = get_ipython()
print("exec count:", ip.execution_count)
n = ip.execution_count - 1
for k in range(max(1, n-2), n+1):
    s = ip.user_ns['In'][k]
    if 'get_ipython' not in s: print(f"--- In[{k}] ---\n{s[:2500]}")
print("=====")
if hasattr(sys, 'last_value'):
    traceback.print_exception(sys.last_type, sys.last_value, sys.last_traceback, limit=-3)

exec count: 17
--- In[14] ---
rint("\n🔗 MERGING SENTIMENT WITH PRICE DATA")
print("="*60)

stock_data_final = {}

for ticker in stocks:
    print(f"\n📊 Processing {ticker}...")

    # Get price data with technical indicators
    df = stock_data_with_features[ticker].copy()

    # Try to load sentiment data
    try:
        sentiment_df = pd.read_csv(f'../data/{ticker}_daily_sentiment.csv')
        sentiment_df['date'] = pd.to_datetime(sentiment_df['date'])
        sentiment_df.set_index('date', inplace=True)

        # Merge on date
        df = df.join(sentiment_df, how='left')

        # Forward fill sentiment (if no news, use previous day's sentiment)
        sentiment_cols = ['avg_sentiment', 'sentiment_std', 'news_count', 
                         'avg_positive', 'avg_negative', 'avg_neutral']
        df[sentiment_cols] = df[sentiment_cols].fillna(method='ffill')

        # Fill remaining NaNs with neutral (0)
        df['avg_sentiment'] = df['avg_sentiment'].fillna(0)
        df[

Traceback (most recent call last):
  File "/Users/ryankhurana/Desktop/Financial_Sentiment_Trading/.venv/lib/python3.13/site-packages/IPython/core/interactiveshell.py", line 3823, in run_code
    exec(code_obj, self.user_global_ns, self.user_ns)
    ~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/var/folders/zd/wq0h2pzd4ks66gj19y5_p7br0000gn/T/ipykernel_73211/4019161794.py", line 24, in <module>
    df[sentiment_cols] = df[sentiment_cols].fillna(method='ffill')
                         ~~~~~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^
TypeError: NDFrame.fillna() got an unexpected keyword argument 'method'


In [ ]:
def _test(ticker):
    df = stock_data_with_features[ticker].copy()
    s = pd.read_csv(f'../data/{ticker}_daily_sentiment.csv')
    s['date'] = pd.to_datetime(s['date'])
    s = s.set_index('date')
    df = df.join(s, how='left')
    cols = ['avg_sentiment', 'sentiment_std', 'news_count', 'avg_positive', 'avg_negative', 'avg_neutral']
    df[cols] = df[cols].ffill()
    return df.shape, int((df['news_count'].fillna(0) > 0).sum())
print({t: _test(t) for t in stocks})
del _test

{'AAPL': ((500, 32), 23), 'TSLA': ((500, 32), 25), 'JPM': ((500, 32), 29), 'AMZN': ((500, 32), 23), 'MSFT': ((500, 32), 24)}


In [ ]:
# Cell 4: Create target variable and lagged features
print("\n🎯 CREATING TARGET VARIABLE & LAGGED FEATURES")
print("="*60)

def prepare_ml_features(df, target_days=1):
    """
    Prepare features for machine learning
    
    target_days: predict price movement N days ahead
    """
    df = df.copy()
    
    # Ensure index is datetime
    if not isinstance(df.index, pd.DatetimeIndex):
        df.index = pd.to_datetime(df.index)
    
    # TARGET: Next day's price direction (1=up, 0=down)
    df['target_price'] = df['Close'].shift(-target_days)
    df['target_return'] = (df['target_price'] - df['Close']) / df['Close']
    df['target_direction'] = (df['target_return'] > 0).astype(int)
    
    # TARGET: Actual return (for regression)
    df['target_return_pct'] = df['target_return'] * 100
    
    # LAGGED FEATURES (use past data to predict future)
    lag_features = ['returns', 'volume_change', 'RSI', 'MACD', 'avg_sentiment']
    
    for feature in lag_features:
        if feature in df.columns:
            # Create 1, 3, 5 day lags
            df[f'{feature}_lag1'] = df[feature].shift(1)
            df[f'{feature}_lag3'] = df[feature].shift(3)
            df[f'{feature}_lag5'] = df[feature].shift(5)
    
    # ROLLING FEATURES (trend indicators)
    df['returns_roll_mean_5'] = df['returns'].rolling(window=5).mean()
    df['returns_roll_std_5'] = df['returns'].rolling(window=5).std()
    df['volume_roll_mean_5'] = df['volume_change'].rolling(window=5).mean()
    
    # Day of week (some days have patterns) - FIX
    df['day_of_week'] = df.index.dayofweek if hasattr(df.index, 'dayofweek') else df.index.to_series().dt.dayofweek.values
    df['month'] = df.index.month if hasattr(df.index, 'month') else df.index.to_series().dt.month.values
    
    return df

# Apply to all stocks
stock_data_ml = {}

for ticker in stocks:
    print(f"\n📊 Preparing ML features for {ticker}...")
    df = prepare_ml_features(stock_data_final[ticker])
    
    # Drop NaN rows (from lagging and technical indicators)
    df_clean = df.dropna()
    
    stock_data_ml[ticker] = df_clean
    
    print(f"   Original rows: {len(stock_data_final[ticker])}")
    print(f"   After cleaning: {len(df_clean)}")
    print(f"   Features: {df_clean.shape[1]}")
    
    # Show target distribution
    target_dist = df_clean['target_direction'].value_counts()
    print(f"   Target distribution:")
    print(f"      Up days: {target_dist.get(1, 0)} ({target_dist.get(1, 0)/len(df_clean)*100:.1f}%)")
    print(f"      Down days: {target_dist.get(0, 0)} ({target_dist.get(0, 0)/len(df_clean)*100:.1f}%)")

print("\n✅ ML feature preparation complete!")


🎯 CREATING TARGET VARIABLE & LAGGED FEATURES

📊 Preparing ML features for AAPL...


KeyError: 'AAPL'

In [ ]:
import sys, traceback
ip = get_ipython()
print("exec count:", ip.execution_count)
n = ip.execution_count - 1
for k in range(max(1, n-3), n+1):
    s = ip.user_ns['In'][k]
    if 'get_ipython' not in s and '_test' not in s: print(f"--- In[{k}] ---\n{s[:3000]}")
print("=====")
if hasattr(sys, 'last_value'):
    traceback.print_exception(sys.last_type, sys.last_value, sys.last_traceback, limit=-3)

exec count: 6
--- In[2] ---
print("\n CALCULATING TECHNICAL INDICATORS")
print("="*60)

def add_technical_indicators(df):
    """Add technical analysis indicators to stock dataframe"""

    # Make a copy
    df = df.copy()

    # 1. RETURNS
    df['returns'] = df['Close'].pct_change()
    df['log_returns'] = np.log(df['Close'] / df['Close'].shift(1))

    # 2. MOVING AVERAGES
    df['MA_7'] = df['Close'].rolling(window=7).mean()
    df['MA_21'] = df['Close'].rolling(window=21).mean()
    df['MA_50'] = df['Close'].rolling(window=50).mean()

    # 3. EXPONENTIAL MOVING AVERAGES
    df['EMA_12'] = df['Close'].ewm(span=12, adjust=False).mean()
    df['EMA_26'] = df['Close'].ewm(span=26, adjust=False).mean()

    # 4. MACD (Moving Average Convergence Divergence)
    df['MACD'] = df['EMA_12'] - df['EMA_26']
    df['MACD_signal'] = df['MACD'].ewm(span=9, adjust=False).mean()

    # 5. BOLLINGER BANDS
    df['BB_middle'] = df['Close'].rolling(window=20).mean()
    df['BB_std'] = df['Close'].ro

Traceback (most recent call last):
  File "/Users/ryankhurana/Desktop/Financial_Sentiment_Trading/.venv/lib/python3.13/site-packages/IPython/core/interactiveshell.py", line 3823, in run_code
    exec(code_obj, self.user_global_ns, self.user_ns)
    ~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/var/folders/zd/wq0h2pzd4ks66gj19y5_p7br0000gn/T/ipykernel_73925/2590634391.py", line 51, in <module>
    df = prepare_ml_features(stock_data_final[ticker])
                             ~~~~~~~~~~~~~~~~^^^^^^^^
KeyError: 'AAPL'


In [ ]:
# Cell 5: Train-test split (time-series split)
print("\n✂️ TRAIN-TEST SPLIT (TIME-SERIES)")
print("="*60)

def time_series_split(df, test_size=0.2):
    """
    Split data chronologically (important for time-series!)
    
    Never use random split for time-series - that's data leakage!
    """
    # Split point
    split_idx = int(len(df) * (1 - test_size))
    
    train = df.iloc[:split_idx]
    test = df.iloc[split_idx:]
    
    return train, test

# Define feature columns (exclude target and non-predictive columns)
exclude_cols = ['target_price', 'target_return', 'target_direction', 'target_return_pct',
                'Open', 'High', 'Low', 'Close', 'Adj Close', 'Volume']

# Split all stocks
train_test_data = {}

for ticker in stocks:
    print(f"\n📊 Splitting {ticker}...")
    
    df = stock_data_ml[ticker]
    
    # Get feature columns
    feature_cols = [col for col in df.columns if col not in exclude_cols]
    
    # Split
    train_df, test_df = time_series_split(df, test_size=0.2)
    
    # Separate X and y
    X_train = train_df[feature_cols]
    y_train_direction = train_df['target_direction']
    y_train_return = train_df['target_return_pct']
    
    X_test = test_df[feature_cols]
    y_test_direction = test_df['target_direction']
    y_test_return = test_df['target_return_pct']
    
    # Store
    train_test_data[ticker] = {
        'X_train': X_train,
        'y_train_direction': y_train_direction,
        'y_train_return': y_train_return,
        'X_test': X_test,
        'y_test_direction': y_test_direction,
        'y_test_return': y_test_return,
        'train_dates': train_df.index,
        'test_dates': test_df.index,
        'feature_cols': feature_cols
    }
    
    print(f"   Training set: {len(X_train)} days ({train_df.index[0].date()} to {train_df.index[-1].date()})")
    print(f"   Test set: {len(X_test)} days ({test_df.index[0].date()} to {test_df.index[-1].date()})")
    print(f"   Features: {len(feature_cols)}")

print("\n✅ Train-test split complete!")
print("\n💡 Key point: We used CHRONOLOGICAL split (not random)")
print("   This prevents data leakage in time-series prediction!")

In [ ]:
print(get_ipython().user_ns['In'][1])

# Cell 1: Import libraries and load stock data
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

print(" Libraries imported!")

# Load stock price data (FIX: ensure numeric columns)
stocks = ['AAPL', 'TSLA', 'JPM', 'AMZN', 'MSFT']
stock_data = {}

for ticker in stocks:
    df = pd.read_csv(f'../data/{ticker}_prices.csv', index_col=0, parse_dates=True)

    # Convert all price/volume columns to numeric
    numeric_cols = ['Open', 'High', 'Low', 'Close', 'Adj Close', 'Volume']
    for col in numeric_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')

    # Drop any rows with NaN after conversion
    df = df.dropna()

    stock_data[ticker] = df
    print(f"✅ Loaded {ticker}: {len(df)} days")

print("\n Starting Feature Engineering!")


In [ ]:
print("="*60)

# Save each stock's processed data
for ticker in stocks:
    data = train_test_data[ticker]
    
    # Save train
    train_combined = pd.DataFrame(data['X_train'])
    train_combined['target_direction'] = data['y_train_direction'].values
    train_combined['target_return'] = data['y_train_return'].values
    train_combined.to_csv(f'../data/{ticker}_train.csv')
    
    # Save test
    test_combined = pd.DataFrame(data['X_test'])
    test_combined['target_direction'] = data['y_test_direction'].values
    test_combined['target_return'] = data['y_test_return'].values
    test_combined.to_csv(f'../data/{ticker}_test.csv')
    
    print(f"✅ Saved {ticker} train/test data")

print("\n" + "="*60)
print("🎉 PHASE 3 COMPLETE: FEATURE ENGINEERING")
print("="*60)

print(f"""
📊 FEATURE ENGINEERING SUMMARY:

TECHNICAL INDICATORS CREATED:
- Moving Averages: MA_7, MA_21, MA_50, EMA_12, EMA_26
- Momentum: MACD, RSI
- Volatility: Bollinger Bands, rolling std
- Volume: volume changes, moving averages
- Returns: daily, log returns

SENTIMENT FEATURES:
- FinBERT sentiment scores (if available)
- News count per day
- Positive/Negative/Neutral probabilities

TARGET VARIABLES:
- target_direction: Binary (up=1, down=0)
- target_return_pct: Continuous (% return)

LAGGED FEATURES:
- 1, 3, 5-day lags for key indicators
- Rolling statistics (5-day windows)
- Temporal features (day of week, month)

DATA SPLIT:
- Chronological split (80% train, 20% test)
- No data leakage (past predicts future)
- {len(stocks)} stocks prepared

TOTAL FEATURES PER STOCK: ~{len(train_test_data[stocks[0]]['feature_cols'])} features

NEXT: Phase 4 - Model Building
- LSTM (PyTorch) for sequential patterns
- LightGBM for feature combinations
- Backtesting & portfolio simulation

In [ ]:
import json as _json
_p = '/private/tmp/claude-501/-Users-ryankhurana-Desktop-Financial-Sentiment-Trading/1ceea934-87d2-4eba-afbd-a72bda15e6f6/scratchpad/user_cells.json'
with open(_p, 'w') as _f:
    _json.dump([get_ipython().user_ns['In'][k] for k in range(1, 5)], _f)
print("saved"); del _json, _p, _f

In [ ]:
import sys, traceback
ip = get_ipython()
print("exec count:", ip.execution_count)
n = ip.execution_count - 1
s = ip.user_ns['In'][n-1] if n > 1 else ''
print(f"--- In[{n-1}] (first 400 chars) ---\n{s[:400]}")
print("=====")
if hasattr(sys, 'last_value'):
    traceback.print_exception(sys.last_type, sys.last_value, sys.last_traceback, limit=-2)

exec count: 17
--- In[15] (first 400 chars) ---
print("\n🔗 MERGING SENTIMENT WITH PRICE DATA")
print("="*60)

stock_data_final = {}

for ticker in stocks:
    print(f"\n📊 Processing {ticker}...")

    # Get price data with technical indicators
    df = stock_data_with_features[ticker].copy()

    # Try to load sentiment data
    try:
        sentiment_df = pd.read_csv(f'../data/{ticker}_daily_sentiment.csv')
        sentiment_df['date'] = pd.to
=====


Traceback (most recent call last):
  File "/Users/ryankhurana/Desktop/Financial_Sentiment_Trading/.venv/lib/python3.13/site-packages/IPython/core/interactiveshell.py", line 3823, in run_code
    exec(code_obj, self.user_global_ns, self.user_ns)
    ~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/var/folders/zd/wq0h2pzd4ks66gj19y5_p7br0000gn/T/ipykernel_73925/4019161794.py", line 24, in <module>
    df[sentiment_cols] = df[sentiment_cols].fillna(method='ffill')
                         ~~~~~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^
TypeError: NDFrame.fillna() got an unexpected keyword argument 'method'


In [ ]:
import sys, traceback
ip = get_ipython()
ins = ip.user_ns['In']
print("exec count:", ip.execution_count)
for k in range(1, len(ins)):
    s = ins[k]
    if s.strip() and 'get_ipython' not in s and '_test' not in s and '_json' not in s:
        print(f"--- In[{k}] | {s.strip().splitlines()[0][:80]} | {len(s)} chars")
print("=====")
if hasattr(sys, 'last_value'):
    traceback.print_exception(sys.last_type, sys.last_value, sys.last_traceback, limit=-2)

exec count: 21
--- In[1] | # Cell 1: Import libraries and load stock data | 923 chars
--- In[2] | print("\n CALCULATING TECHNICAL INDICATORS") | 2394 chars
--- In[3] | print("\n🔗 MERGING SENTIMENT WITH PRICE DATA") | 2029 chars
--- In[4] | # Cell 4: Create target variable and lagged features | 2598 chars
--- In[7] | # Cell 1: Import libraries and load stock data | 923 chars
--- In[8] | print("\n CALCULATING TECHNICAL INDICATORS") | 2394 chars
--- In[9] | print("\n🔗 MERGING SENTIMENT WITH PRICE DATA") | 2029 chars
--- In[10] | # Cell 1: Import libraries and load stock data | 923 chars
--- In[11] | print("\n CALCULATING TECHNICAL INDICATORS") | 2394 chars
--- In[12] | print("\n🔗 MERGING SENTIMENT WITH PRICE DATA") | 2029 chars
--- In[13] | # Cell 1: Import libraries and load stock data | 923 chars
--- In[14] | print("\n CALCULATING TECHNICAL INDICATORS") | 2394 chars
--- In[15] | print("\n🔗 MERGING SENTIMENT WITH PRICE DATA") | 2029 chars
--- In[17] | # Cell 1: Import libraries and load 

Traceback (most recent call last):
  File "/Users/ryankhurana/Desktop/Financial_Sentiment_Trading/.venv/lib/python3.13/site-packages/IPython/core/interactiveshell.py", line 3823, in run_code
    exec(code_obj, self.user_global_ns, self.user_ns)
    ~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/var/folders/zd/wq0h2pzd4ks66gj19y5_p7br0000gn/T/ipykernel_73925/4019161794.py", line 24, in <module>
    df[sentiment_cols] = df[sentiment_cols].fillna(method='ffill')
                         ~~~~~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^
TypeError: NDFrame.fillna() got an unexpected keyword argument 'method'


In [ ]:
import sys, traceback
ip = get_ipython()
ins = ip.user_ns['In']
print("exec count:", ip.execution_count)
for k in range(max(1, len(ins)-6), len(ins)):
    s = ins[k]
    if s.strip() and 'get_ipython' not in s:
        print(f"--- In[{k}] | {s.strip().splitlines()[0][:80]} | {len(s)} chars | ffill fixed: {'.ffill()' in s}")
print("=====")
if hasattr(sys, 'last_value'):
    traceback.print_exception(sys.last_type, sys.last_value, sys.last_traceback, limit=-2)

exec count: 25
--- In[19] | print("\n🔗 MERGING SENTIMENT WITH PRICE DATA") | 2029 chars | ffill fixed: False
--- In[21] | # Cell 1: Import libraries and load stock data | 923 chars | ffill fixed: False
--- In[22] | print("\n CALCULATING TECHNICAL INDICATORS") | 2394 chars | ffill fixed: False
--- In[23] | print("\n🔗 MERGING SENTIMENT WITH PRICE DATA") | 2029 chars | ffill fixed: False
=====


Traceback (most recent call last):
  File "/Users/ryankhurana/Desktop/Financial_Sentiment_Trading/.venv/lib/python3.13/site-packages/IPython/core/interactiveshell.py", line 3823, in run_code
    exec(code_obj, self.user_global_ns, self.user_ns)
    ~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/var/folders/zd/wq0h2pzd4ks66gj19y5_p7br0000gn/T/ipykernel_73925/4019161794.py", line 24, in <module>
    df[sentiment_cols] = df[sentiment_cols].fillna(method='ffill')
                         ~~~~~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^
TypeError: NDFrame.fillna() got an unexpected keyword argument 'method'


In [ ]:
# Fixed Cell 3: Merge sentiment with price data
stock_data_final = {}
sentiment_cols = ['avg_sentiment', 'sentiment_std', 'news_count',
                  'avg_positive', 'avg_negative', 'avg_neutral']
for ticker in stocks:
    df = stock_data_with_features[ticker].copy()
    try:
        sentiment_df = pd.read_csv(f'../data/{ticker}_daily_sentiment.csv')
        sentiment_df['date'] = pd.to_datetime(sentiment_df['date'])
        sentiment_df = sentiment_df.set_index('date')
        df = df.join(sentiment_df[sentiment_cols], how='left')
        df['news_count'] = df['news_count'].fillna(0)
        score_cols = ['avg_sentiment', 'sentiment_std', 'avg_positive', 'avg_negative', 'avg_neutral']
        df[score_cols] = df[score_cols].ffill()
        df['avg_sentiment'] = df['avg_sentiment'].fillna(0)
        df['sentiment_std'] = df['sentiment_std'].fillna(0)
        df['avg_positive'] = df['avg_positive'].fillna(0.33)
        df['avg_negative'] = df['avg_negative'].fillna(0.33)
        df['avg_neutral'] = df['avg_neutral'].fillna(0.34)
    except FileNotFoundError:
        df['avg_sentiment'] = 0; df['sentiment_std'] = 0; df['news_count'] = 0
        df['avg_positive'] = 0.33; df['avg_negative'] = 0.33; df['avg_neutral'] = 0.34
    df.index.name = 'Date'
    stock_data_final[ticker] = df
    print(f"✅ {ticker}: merged, days with news = {int((df['news_count'] > 0).sum())}")

# Fixed Cell 4: Target and lagged features
def prepare_ml_features(df, target_days=1):
    df = df.copy()
    if not isinstance(df.index, pd.DatetimeIndex):
        df.index = pd.to_datetime(df.index)
    df['target_price'] = df['Close'].shift(-target_days)
    df['target_return'] = (df['target_price'] - df['Close']) / df['Close']
    df['target_direction'] = (df['target_return'] > 0).astype(int)
    df['target_return_pct'] = df['target_return'] * 100
    for feature in ['returns', 'volume_change', 'RSI', 'MACD', 'avg_sentiment']:
        if feature in df.columns:
            for lag in (1, 3, 5):
                df[f'{feature}_lag{lag}'] = df[feature].shift(lag)
    df['returns_roll_mean_5'] = df['returns'].rolling(5).mean()
    df['returns_roll_std_5'] = df['returns'].rolling(5).std()
    df['volume_roll_mean_5'] = df['volume_change'].rolling(5).mean()
    df['day_of_week'] = df.index.dayofweek
    df['month'] = df.index.month
    return df

stock_data_ml = {}
for ticker in stocks:
    df_clean = prepare_ml_features(stock_data_final[ticker]).replace([np.inf, -np.inf], np.nan).dropna()
    stock_data_ml[ticker] = df_clean
    print(f"✅ {ticker}: {len(df_clean)} rows, up days {df_clean['target_direction'].mean():.1%}")

# Fixed Cell 5: Combine and save for train_model.ipynb
all_frames = []
for ticker in stocks:
    df = stock_data_ml[ticker].rename(columns={'target_direction': 'target', 'target_return': 'next_day_return'})
    df = df.drop(columns=['target_price', 'target_return_pct'])
    df['ticker'] = ticker
    all_frames.append(df)
features_df = pd.concat(all_frames).reset_index()
features_df.to_csv('../data/features_all.csv', index=False)
print(f"\n✅ Saved ../data/features_all.csv → {features_df.shape[0]} rows × {features_df.shape[1]} columns")

✅ AAPL: merged, days with news = 1
✅ TSLA: merged, days with news = 3
✅ JPM: merged, days with news = 7
✅ AMZN: merged, days with news = 1
✅ MSFT: merged, days with news = 2
✅ AAPL: 450 rows, up days 52.9%
✅ TSLA: 450 rows, up days 50.0%
✅ JPM: 450 rows, up days 55.1%
✅ AMZN: 450 rows, up days 50.4%
✅ MSFT: 450 rows, up days 51.8%

✅ Saved ../data/features_all.csv → 2250 rows × 56 columns
